# ds003645 — MEG vs EEG Manifold Comparison

Wakeman-Henson simultaneous MEEG dataset (face perception task).  
**18 subjects × 6 runs**, Neuromag FIF (MEG + EEG) + standalone EEG .set files.

## Goals
1. Extract and save all H5 features to CSV/JSON for downstream use
2. MEG vs EEG band correlation on paired FIF epochs (same time windows)
3. EEG-derived pseudo-3D trajectory via PCA, compared with MEG-derived MNPS 3D
4. Jacobian eigenvalue spectra comparison
5. Temporal null test (shuffled window order)

## Physical note
MEG and EEG are complementary projections of the same neural sources.  
MEG is maximally sensitive to tangential sources (sulci), EEG captures both tangential and radial sources.  
We expect **moderate** correlation (ρ ~ 0.3–0.7) and **topological** rather than coordinate-level agreement.

## H5 structure (per run)
- `features_raw/names` (49,), `features_raw/values` (N, 49)
- `features_robust_z/values` (N, 49)  — log10+robust_z transformed
- `coords_9d/names` (9,), `coords_9d/values` (N, 9)  — MEG-derived subcoords
- `mnps_3d` (N, 3)  — combined 3D manifold coordinates
- `jacobian/J_hat` (K, 3, 3)  — local Jacobians at K window centers
- `jacobian/centers` (K,)  — epoch indices for Jacobian centers
- `time` (N,), `window_start` (N,), `window_end` (N,)
- `qc/windows/retained_after_qc` (N,)

In [1]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import json
import warnings
from scipy.stats import pearsonr, spearmanr
from scipy.linalg import eigvals
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler

# Plotting (optional — comment out if running headless)
import matplotlib
matplotlib.use('Agg')  # change to 'TkAgg' or 'Qt5Agg' if interactive
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=RuntimeWarning)

print('Imports OK')

Imports OK


In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
OUT_DIR   = Path(r'E:/Science_Datasets/openneuro/processed/ds003645')
RUN_DIR   = OUT_DIR / 'neuralmanifolddynamics_ds003645_20260625_152559'
SAVE_DIR  = OUT_DIR / 'meg_eeg_comparison'
SAVE_DIR.mkdir(exist_ok=True)
FIG_DIR   = SAVE_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

# ── Subjects / runs ────────────────────────────────────────────────────────
# Pilot: sub-002 to sub-006.  Set ALL_SUBS=True to include all 18 subjects.
PILOT_SUBS = ['002', '003', '004', '005', '006']
ALL_SUBS   = [f'{i:03d}' for i in range(2, 20)]  # 002..019
SUBS       = PILOT_SUBS   # ← change to ALL_SUBS for full cohort
RUNS       = ['1', '2', '3', '4', '5', '6']
TASK       = 'FacePerception'

def h5_path(sub, run):
    stem = f'sub-{sub}_meeg_{TASK}_run-{run}'
    return RUN_DIR / stem / f'{stem}.h5'

print('SAVE_DIR:', SAVE_DIR)
print('Expected H5 files:', sum(h5_path(s, r).exists() for s in SUBS for r in RUNS))

SAVE_DIR: E:\Science_Datasets\openneuro\processed\ds003645\meg_eeg_comparison
Expected H5 files: 30


## 1 · Load all H5 files and export features to CSV

In [3]:
def load_h5(path):
    """Load key arrays from one NMD H5 file.  Returns a dict."""
    out = {}
    with h5py.File(path, 'r') as f:
        # Features
        out['feat_names']    = [n.decode() for n in f['features_raw/names'][:]]
        out['feat_raw']      = f['features_raw/values'][:].astype(np.float64)
        out['feat_robustz']  = f['features_robust_z/values'][:].astype(np.float64)
        # 9-D and 3-D coordinates
        out['coord_names']   = [n.decode() for n in f['coords_9d/names'][:]]
        out['coords_9d']     = f['coords_9d/values'][:].astype(np.float64)
        out['mnps_3d']       = f['mnps_3d'][:].astype(np.float64)          # stored directly
        # Jacobians
        out['J_hat']         = f['jacobian/J_hat'][:].astype(np.float64)   # (K, 3, 3)
        out['J_centers']     = f['jacobian/centers'][:]
        # Time
        out['time']          = f['time'][:]
        out['window_start']  = f['window_start'][:]
        out['window_end']    = f['window_end'][:]
        # QC
        out['qc_ok']         = f['qc/windows/retained_after_qc'][:].astype(bool)
    return out


records = {}   # key → dict from load_h5
for sub in SUBS:
    for run in RUNS:
        p = h5_path(sub, run)
        if not p.exists():
            print(f'  MISSING  sub-{sub} run-{run}')
            continue
        d = load_h5(p)
        records[f'sub-{sub}_run-{run}'] = d

print(f'\nLoaded {len(records)} H5 files')


Loaded 30 H5 files


In [4]:
# ── Flatten all features to one big DataFrame and save ───────────────────
rows = []
ref_key  = next(iter(records))
feat_cols = records[ref_key]['feat_names']

for key, d in records.items():
    sub_id, run_id = key.replace('sub-', '').split('_run-')
    n = d['feat_raw'].shape[0]
    meta = pd.DataFrame({
        'sub':           sub_id,
        'run':           run_id,
        'epoch_idx':     np.arange(n),
        'time_s':        d['time'],
        'window_start':  d['window_start'],
        'window_end':    d['window_end'],
        'qc_ok':         d['qc_ok'].astype(int),
    })
    feat_df    = pd.DataFrame(d['feat_raw'],     columns=feat_cols)
    robz_df    = pd.DataFrame(d['feat_robustz'], columns=[c + '_z' for c in feat_cols])
    coord9_df  = pd.DataFrame(d['coords_9d'],    columns=d['coord_names'])
    coord3_df  = pd.DataFrame(d['mnps_3d'],      columns=['mnps_3d_0', 'mnps_3d_1', 'mnps_3d_2'])
    row = pd.concat([meta, feat_df, robz_df, coord9_df, coord3_df], axis=1)
    rows.append(row)

all_df = pd.concat(rows, ignore_index=True)
out_csv = SAVE_DIR / 'all_epochs_features.csv'
all_df.to_csv(out_csv, index=False)
print(f'Saved: {out_csv}  shape={all_df.shape}')

Saved: E:\Science_Datasets\openneuro\processed\ds003645\meg_eeg_comparison\all_epochs_features.csv  shape=(7366, 117)


In [5]:
# ── Flag FIF vs .set source rows (FIF = MEG non-NaN) ──────────────────────
all_df['source'] = np.where(all_df['meg_delta'].notna(), 'fif', 'set')

print(all_df.groupby(['sub', 'source']).size().unstack(fill_value=0))

# Save source-tagged version
all_df.to_csv(SAVE_DIR / 'all_epochs_features.csv', index=False)

# ── Paired (FIF) epochs only ───────────────────────────────────────────────
fif_df = all_df[all_df['source'] == 'fif'].copy()
set_df = all_df[all_df['source'] == 'set'].copy()
print(f'\nFIF epochs (paired MEG+EEG): {len(fif_df)}')
print(f'.set epochs (EEG-only):       {len(set_df)}')

source  fif  set
sub             
002     740  740
003     733  733
004     741  741
005     735  735
006     734  734



FIF epochs (paired MEG+EEG): 3683
.set epochs (EEG-only):       3683


In [6]:
# ── Save Jacobians to JSON (per run) ─────────────────────────────────────
jac_records = []
for key, d in records.items():
    sub_id, run_id = key.replace('sub-', '').split('_run-')
    J = d['J_hat']  # (K, 3, 3)
    centers = d['J_centers']
    for k, (c, j) in enumerate(zip(centers, J)):
        evs = np.linalg.eigvals(j)
        jac_records.append({
            'sub': sub_id, 'run': run_id,
            'window_center_idx': int(c),
            'time_s': float(d['time'][c]) if c < len(d['time']) else None,
            'ev_real_0': float(evs[0].real), 'ev_real_1': float(evs[1].real), 'ev_real_2': float(evs[2].real),
            'ev_imag_0': float(evs[0].imag), 'ev_imag_1': float(evs[1].imag), 'ev_imag_2': float(evs[2].imag),
            'ev_max_real': float(np.max(evs.real)),
            'spectral_radius': float(np.max(np.abs(evs))),
            'J_flat': j.flatten().tolist(),
        })

jac_df = pd.DataFrame(jac_records)
jac_df.to_csv(SAVE_DIR / 'jacobian_eigenvalues.csv', index=False)
print(f'Saved jacobian_eigenvalues.csv  shape={jac_df.shape}')

Saved jacobian_eigenvalues.csv  shape=(3662, 13)


## 2 · MEG vs EEG band correlation (paired FIF epochs)

In [7]:
BANDS = ['delta', 'theta', 'alpha', 'beta', 'gamma']
EXTRAS = ['hjorth_mobility', 'hjorth_complexity', 'permutation_entropy', 'alpha_theta', 'beta_alpha']
ALL_FEATS = BANDS + EXTRAS

corr_rows = []
for (sub, run), grp in fif_df.groupby(['sub', 'run']):
    qc = grp['qc_ok'].astype(bool)
    grp_qc = grp[qc]
    for feat in ALL_FEATS:
        meg_col = f'meg_{feat}'
        eeg_col = f'eeg_{feat}'
        if meg_col not in grp_qc or eeg_col not in grp_qc:
            continue
        meg_v = grp_qc[meg_col].to_numpy(dtype=float)
        eeg_v = grp_qc[eeg_col].to_numpy(dtype=float)
        valid = np.isfinite(meg_v) & np.isfinite(eeg_v)
        if valid.sum() < 10:
            continue
        # Log-transform band powers before correlating (they span very different physical ranges)
        if feat in BANDS:
            meg_x = np.log10(meg_v[valid] + 1e-40)
            eeg_x = np.log10(eeg_v[valid] + 1e-30)
        else:
            meg_x = meg_v[valid]
            eeg_x = eeg_v[valid]
        r_p, p_p = pearsonr(meg_x, eeg_x)
        r_s, p_s = spearmanr(meg_x, eeg_x)
        corr_rows.append({
            'sub': sub, 'run': run, 'feature': feat,
            'n': int(valid.sum()),
            'pearson_r': r_p, 'pearson_p': p_p,
            'spearman_r': r_s, 'spearman_p': p_s,
        })

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv(SAVE_DIR / 'meg_eeg_feature_correlations.csv', index=False)

# Summary per feature
summary = corr_df.groupby('feature')[['pearson_r', 'spearman_r']].agg(['mean', 'std']).round(3)
print(summary.to_string())

                    pearson_r        spearman_r       
                         mean    std       mean    std
feature                                               
alpha                   0.231  0.194      0.229  0.193
alpha_theta             0.078  0.161      0.077  0.144
beta                    0.147  0.220      0.123  0.231
beta_alpha              0.089  0.143      0.091  0.139
delta                   0.099  0.113      0.134  0.123
gamma                   0.214  0.210      0.201  0.200
hjorth_complexity       0.025  0.154      0.038  0.139
hjorth_mobility         0.039  0.153      0.031  0.134
permutation_entropy    -0.002  0.152     -0.001  0.131
theta                   0.031  0.135      0.033  0.136


In [8]:
# ── Plot: MEG vs EEG Pearson r per band ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
feat_order = ALL_FEATS
feat_means = corr_df.groupby('feature')['pearson_r'].mean().reindex(feat_order)
feat_sems  = corr_df.groupby('feature')['pearson_r'].sem().reindex(feat_order)
x = np.arange(len(feat_order))

ax.bar(x, feat_means, yerr=feat_sems, capsize=4, color='steelblue', alpha=0.8)
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(feat_order, rotation=30, ha='right')
ax.set_ylabel('Pearson r  (MEG vs EEG, log10 band powers)')
ax.set_title('MEG–EEG feature correlation — ds003645 pilot (sub-002..006, all runs)')
ax.set_ylim(-0.4, 1.0)
fig.tight_layout()
fig.savefig(FIG_DIR / 'meg_eeg_feature_correlation.png', dpi=150)
plt.close(fig)
print('Saved figure: meg_eeg_feature_correlation.png')

Saved figure: meg_eeg_feature_correlation.png


## 3 · EEG-derived pseudo-3D trajectory (PCA of EEG features)

The MNPS 3D is derived from MEG features only (via the 9D subcoords).  
Here we compute an independent EEG-derived 3D embedding using PCA on EEG robust-z features,
then compare its trajectory with the MEG-derived MNPS 3D.

In [9]:
EEG_FEAT_COLS = [f'eeg_{f}' for f in ALL_FEATS]
MEG_FEAT_COLS = [f'meg_{f}' for f in ALL_FEATS]

# Use only FIF epochs (both modalities present) that pass QC
paired = fif_df[fif_df['qc_ok'] == 1].copy()
eeg_robz_cols = [c + '_z' for c in EEG_FEAT_COLS]
meg_robz_cols = [c + '_z' for c in MEG_FEAT_COLS]

available_eeg = [c for c in eeg_robz_cols if c in paired.columns]
available_meg = [c for c in meg_robz_cols if c in paired.columns]

eeg_mat = paired[available_eeg].to_numpy(dtype=float)
meg_mat = paired[available_meg].to_numpy(dtype=float)

# Drop rows with any NaN in either modality
valid_mask = np.isfinite(eeg_mat).all(axis=1) & np.isfinite(meg_mat).all(axis=1)
eeg_clean  = eeg_mat[valid_mask]
meg_clean  = meg_mat[valid_mask]

print(f'Paired QC-passed epochs used for PCA: {valid_mask.sum()}')

# PCA separately for EEG and MEG
pca_eeg = PCA(n_components=3, random_state=42)
pca_meg = PCA(n_components=3, random_state=42)
eeg_3d  = pca_eeg.fit_transform(eeg_clean)
meg_3d  = pca_meg.fit_transform(meg_clean)

print(f'EEG PCA explained variance (3 PCs): {pca_eeg.explained_variance_ratio_.cumsum()[-1]:.3f}')
print(f'MEG PCA explained variance (3 PCs): {pca_meg.explained_variance_ratio_.cumsum()[-1]:.3f}')

# Retrieve the MNPS 3D for the same epochs
mnps3d_cols = ['mnps_3d_0', 'mnps_3d_1', 'mnps_3d_2']
paired_valid_idx = paired.index[valid_mask]
mnps_3d_paired = paired.loc[paired_valid_idx, mnps3d_cols].to_numpy(dtype=float)

Paired QC-passed epochs used for PCA: 3683
EEG PCA explained variance (3 PCs): 0.851
MEG PCA explained variance (3 PCs): 0.899


In [10]:
# ── Per-component correlation: EEG-PCA vs MEG-PCA (raw), and EEG-PCA vs MNPS-3D
from scipy.spatial import procrustes as procrustes_scipy

pca_results = {}
for i in range(3):
    r_em, p_em = pearsonr(eeg_3d[:, i], meg_3d[:, i])
    r_en, p_en = pearsonr(eeg_3d[:, i], mnps_3d_paired[:, i])
    pca_results[f'PC{i+1}'] = {
        'eeg_vs_meg_pca_r': round(r_em, 3), 'eeg_vs_meg_pca_p': round(p_em, 5),
        'eeg_vs_mnps3d_r':  round(r_en, 3), 'eeg_vs_mnps3d_p':  round(p_en, 5),
    }

print(json.dumps(pca_results, indent=2))

# ── Procrustes similarity after optimal rotation ──────────────────────────
# Standardise both clouds to unit Frobenius norm
def normalise_cloud(X):
    X = X - X.mean(axis=0)
    X = X / np.sqrt((X**2).sum())
    return X

_, _, proc_disparity_eeg_meg  = procrustes_scipy(normalise_cloud(meg_3d),  normalise_cloud(eeg_3d))
_, _, proc_disparity_eeg_mnps = procrustes_scipy(normalise_cloud(mnps_3d_paired), normalise_cloud(eeg_3d))

proc_sim_eeg_meg  = 1.0 - proc_disparity_eeg_meg
proc_sim_eeg_mnps = 1.0 - proc_disparity_eeg_mnps

print(f'\nProcrustes similarity  EEG-PCA vs MEG-PCA : {proc_sim_eeg_meg:.3f}')
print(f'Procrustes similarity  EEG-PCA vs MNPS-3D : {proc_sim_eeg_mnps:.3f}')

# Save
pca_summary = {
    'eeg_pca_explained_var_3pc': float(pca_eeg.explained_variance_ratio_.cumsum()[-1]),
    'meg_pca_explained_var_3pc': float(pca_meg.explained_variance_ratio_.cumsum()[-1]),
    'n_paired_epochs': int(valid_mask.sum()),
    'per_component': pca_results,
    'procrustes_similarity_eeg_vs_meg_pca':  round(proc_sim_eeg_meg, 4),
    'procrustes_similarity_eeg_vs_mnps3d':   round(proc_sim_eeg_mnps, 4),
}
with open(SAVE_DIR / 'pca_comparison_summary.json', 'w') as fh:
    json.dump(pca_summary, fh, indent=2)
print('\nSaved pca_comparison_summary.json')

{
  "PC1": {
    "eeg_vs_meg_pca_r": -0.05,
    "eeg_vs_meg_pca_p": 0.0024,
    "eeg_vs_mnps3d_r": -0.032,
    "eeg_vs_mnps3d_p": 0.04921
  },
  "PC2": {
    "eeg_vs_meg_pca_r": 0.059,
    "eeg_vs_meg_pca_p": 0.00034,
    "eeg_vs_mnps3d_r": 0.042,
    "eeg_vs_mnps3d_p": 0.01093
  },
  "PC3": {
    "eeg_vs_meg_pca_r": -0.006,
    "eeg_vs_meg_pca_p": 0.73615,
    "eeg_vs_mnps3d_r": -0.003,
    "eeg_vs_mnps3d_p": 0.85637
  }
}

Procrustes similarity  EEG-PCA vs MEG-PCA : 0.003
Procrustes similarity  EEG-PCA vs MNPS-3D : 0.002

Saved pca_comparison_summary.json


In [11]:
# ── Save 3D trajectories to CSV for later plotting ────────────────────────
paired_meta = paired.loc[paired_valid_idx, ['sub', 'run', 'epoch_idx', 'time_s', 'qc_ok']].reset_index(drop=True)

traj_df = pd.concat([
    paired_meta,
    pd.DataFrame(eeg_3d,  columns=['eeg_pc0', 'eeg_pc1', 'eeg_pc2']),
    pd.DataFrame(meg_3d,  columns=['meg_pc0', 'meg_pc1', 'meg_pc2']),
    pd.DataFrame(mnps_3d_paired, columns=['mnps_0',  'mnps_1',  'mnps_2']),
], axis=1)

traj_df.to_csv(SAVE_DIR / 'trajectories_3d.csv', index=False)
print(f'Saved trajectories_3d.csv  shape={traj_df.shape}')

Saved trajectories_3d.csv  shape=(3683, 14)


In [12]:
# ── Plot: MNPS-3D vs EEG-PCA-3D, first two components, colored by subject
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
subs_uniq = sorted(traj_df['sub'].unique())
cmap = plt.cm.tab10
colors = {s: cmap(i / max(len(subs_uniq) - 1, 1)) for i, s in enumerate(subs_uniq)}

for sub_id, grp in traj_df.groupby('sub'):
    c = colors[sub_id]
    axes[0].scatter(grp['mnps_0'], grp['mnps_1'], s=4, color=c, alpha=0.5, label=f'sub-{sub_id}')
    axes[1].scatter(grp['eeg_pc0'], grp['eeg_pc1'], s=4, color=c, alpha=0.5)

axes[0].set_title('MEG MNPS-3D  (PC1 vs PC2)')
axes[1].set_title('EEG-PCA 3D  (PC1 vs PC2)')
for ax in axes:
    ax.set_xlabel('Component 0')
    ax.set_ylabel('Component 1')
axes[0].legend(markerscale=3, fontsize=7)
fig.suptitle('MEG vs EEG trajectory — ds003645 pilot')
fig.tight_layout()
fig.savefig(FIG_DIR / 'meg_eeg_trajectories_2d.png', dpi=150)
plt.close(fig)
print('Saved figure: meg_eeg_trajectories_2d.png')

Saved figure: meg_eeg_trajectories_2d.png


## 4 · Jacobian eigenvalue analysis

In [13]:
# ── Eigenvalue spectra from the MEG-derived Jacobians ────────────────────
# Note: MEG and EEG share the SAME Jacobian (it's computed on the combined MNPS 3D).
# What we can do: compare J_hat per subject / per run for consistency.

ev_real_cols = ['ev_real_0', 'ev_real_1', 'ev_real_2']
ev_imag_cols = ['ev_imag_0', 'ev_imag_1', 'ev_imag_2']

# Per-subject eigenvalue statistics
jac_summary_rows = []
for sub, grp in jac_df.groupby('sub'):
    ev_real = grp[ev_real_cols].to_numpy()
    ev_imag = grp[ev_imag_cols].to_numpy()
    jac_summary_rows.append({
        'sub': sub,
        'n_windows': len(grp),
        'spectral_radius_mean': grp['spectral_radius'].mean(),
        'spectral_radius_std':  grp['spectral_radius'].std(),
        'max_real_mean':        grp['ev_max_real'].mean(),
        'max_real_std':         grp['ev_max_real'].std(),
        'frac_real_eigvals':    np.mean(np.abs(ev_imag) < 1e-8),
    })

jac_summary_df = pd.DataFrame(jac_summary_rows)
jac_summary_df.to_csv(SAVE_DIR / 'jacobian_subject_summary.csv', index=False)
print(jac_summary_df.to_string(index=False))

sub  n_windows  spectral_radius_mean  spectral_radius_std  max_real_mean  max_real_std  frac_real_eigvals
002        738              0.069703             0.079752       0.036051      0.081460           0.486902
003        727              0.064005             0.038858       0.033360      0.038902           0.502980
004        736              0.061639             0.029689       0.030821      0.033425           0.457428
005        730              0.066114             0.032566       0.035750      0.033874           0.497717
006        731              0.069379             0.036904       0.039291      0.038458           0.472868


In [14]:
# ── Plot: spectral radius distribution per subject ────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
subs_jac = sorted(jac_df['sub'].unique())
data_to_plot = [jac_df[jac_df['sub'] == s]['spectral_radius'].dropna().to_numpy() for s in subs_jac]
ax.boxplot(data_to_plot, tick_labels=[f's{s}' for s in subs_jac], patch_artist=True)
ax.set_ylabel('Jacobian spectral radius')
ax.set_title('J_hat spectral radius per subject — ds003645')
ax.axhline(1.0, color='r', ls='--', lw=0.8, label='neutral (=1)')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'jacobian_spectral_radius.png', dpi=150)
plt.close(fig)
print('Saved figure: jacobian_spectral_radius.png')

Saved figure: jacobian_spectral_radius.png

## 5 · Temporal null test

Shuffle epoch order within each run (breaking temporal structure) and recompute MEG–EEG feature correlations.  
If the true correlation exceeds the shuffled distribution, the correlation reflects genuine temporal covariation, not a static bias.

In [15]:
N_SHUFFLES = 200
rng = np.random.default_rng(42)

null_rows = []
for feat in BANDS:                  # focus on band powers for the null test
    meg_col = f'meg_{feat}_z'
    eeg_col = f'eeg_{feat}_z'
    if meg_col not in fif_df.columns or eeg_col not in fif_df.columns:
        continue

    # Observed correlation (across all paired QC-passed epochs)
    sub_df = fif_df[fif_df['qc_ok'] == 1]
    meg_v  = sub_df[meg_col].to_numpy(dtype=float)
    eeg_v  = sub_df[eeg_col].to_numpy(dtype=float)
    valid  = np.isfinite(meg_v) & np.isfinite(eeg_v)
    if valid.sum() < 20:
        continue
    r_obs, _ = pearsonr(meg_v[valid], eeg_v[valid])

    # Shuffle within each run separately (preserves marginal distributions)
    null_rs = []
    for _ in range(N_SHUFFLES):
        shuffled_meg = np.empty_like(meg_v)
        for _, run_grp in sub_df.groupby('run'):
            idx = run_grp.index
            perm = rng.permutation(len(idx))
            shuffled_meg[np.isin(sub_df.index, idx)] = run_grp[meg_col].to_numpy()[perm]
        v2 = np.isfinite(shuffled_meg) & np.isfinite(eeg_v)
        if v2.sum() < 20:
            continue
        r_null, _ = pearsonr(shuffled_meg[v2], eeg_v[v2])
        null_rs.append(r_null)

    null_arr = np.array(null_rs)
    p_null   = np.mean(null_arr >= r_obs)
    drop_abs = r_obs - null_arr.mean()
    null_rows.append({
        'feature': feat,
        'r_observed': round(r_obs, 4),
        'null_mean':  round(null_arr.mean(), 4),
        'null_std':   round(null_arr.std(), 4),
        'drop_abs':   round(drop_abs, 4),
        'p_null':     round(p_null, 4),
        'n_shuffles': len(null_rs),
    })

null_df = pd.DataFrame(null_rows)
null_df.to_csv(SAVE_DIR / 'temporal_null_test.csv', index=False)
print(null_df.to_string(index=False))

feature  r_observed  null_mean  null_std  drop_abs  p_null  n_shuffles
  delta      0.0053     0.0051    0.0192    0.0001   0.345         200
  theta      0.0088     0.0031    0.0164    0.0057   0.235         200
  alpha      0.0532     0.0025    0.0250    0.0507   0.015         200
   beta      0.0671     0.0033    0.0195    0.0639   0.015         200
  gamma      0.1548     0.0023    0.0152    0.1525   0.000         200


In [16]:
# ── Plot: observed r vs null distribution per band ────────────────────────
fig, axes = plt.subplots(1, len(null_df), figsize=(3 * len(null_df), 4), sharey=False)
if len(null_df) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, null_df.iterrows()):
    ax.axvline(row['r_observed'], color='crimson', lw=2, label=f"r_obs={row['r_observed']:.3f}")
    ax.axvline(row['null_mean'],  color='steelblue', lw=1.5, ls='--', label=f"null={row['null_mean']:.3f}")
    ax.set_title(row['feature'])
    ax.set_xlabel('Pearson r')
    ax.legend(fontsize=7)
    ax.set_xlim(-0.3, 0.9)

fig.suptitle('Temporal null test — MEG vs EEG band correlation  (n=200 shuffles per band)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'temporal_null_test.png', dpi=150)
plt.close(fig)
print('Saved figure: temporal_null_test.png')

Saved figure: temporal_null_test.png


## 6 · Summary export

In [17]:
summary = {
    'dataset': 'ds003645',
    'task': TASK,
    'subjects_analysed': SUBS,
    'runs_analysed': RUNS,
    'n_h5_files_loaded': len(records),
    'n_fif_epochs_total': int((all_df['source'] == 'fif').sum()),
    'n_set_epochs_total': int((all_df['source'] == 'set').sum()),
    'meg_eeg_band_correlation_mean': {
        f: round(corr_df[corr_df['feature'] == f]['pearson_r'].mean(), 3)
        for f in BANDS if f in corr_df['feature'].values
    },
    'procrustes_similarity_eeg_vs_mnps3d': pca_summary['procrustes_similarity_eeg_vs_mnps3d'],
    'temporal_null_results': null_df.set_index('feature')[['r_observed', 'null_mean', 'p_null']].to_dict('index'),
    'output_files': [
        'all_epochs_features.csv',
        'jacobian_eigenvalues.csv',
        'jacobian_subject_summary.csv',
        'meg_eeg_feature_correlations.csv',
        'trajectories_3d.csv',
        'pca_comparison_summary.json',
        'temporal_null_test.csv',
        'figures/meg_eeg_feature_correlation.png',
        'figures/meg_eeg_trajectories_2d.png',
        'figures/jacobian_spectral_radius.png',
        'figures/temporal_null_test.png',
    ],
}

with open(SAVE_DIR / 'analysis_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2)

print('=== Final summary ===')
print(json.dumps(summary, indent=2))
print(f'\nAll outputs in: {SAVE_DIR}')

=== Final summary ===
{
  "dataset": "ds003645",
  "task": "FacePerception",
  "subjects_analysed": [
    "002",
    "003",
    "004",
    "005",
    "006"
  ],
  "runs_analysed": [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6"
  ],
  "n_h5_files_loaded": 30,
  "n_fif_epochs_total": 3683,
  "n_set_epochs_total": 3683,
  "meg_eeg_band_correlation_mean": {
    "delta": 0.099,
    "theta": 0.031,
    "alpha": 0.231,
    "beta": 0.147,
    "gamma": 0.214
  },
  "procrustes_similarity_eeg_vs_mnps3d": 0.0021,
  "temporal_null_results": {
    "delta": {
      "r_observed": 0.0053,
      "null_mean": 0.0051,
      "p_null": 0.345
    },
    "theta": {
      "r_observed": 0.0088,
      "null_mean": 0.0031,
      "p_null": 0.235
    },
    "alpha": {
      "r_observed": 0.0532,
      "null_mean": 0.0025,
      "p_null": 0.015
    },
    "beta": {
      "r_observed": 0.0671,
      "null_mean": 0.0033,
      "p_null": 0.015
    },
    "gamma": {
      "r_observed": 0.1548,
      "null_mean"

## 7 · Face vs Scrambled contrast

Each 8-second window (4s step) contains ~4 face stimuli.  
Epoch labels: face (famous+unfamiliar) / scrambled / mixed / no_stim.  
Permutation test: shuffle labels within each run, N=2000.

In [18]:
# 7a: Load events and label epochs
DATA_ROOT = Path(r'K:/ExternalReceivedDatasets/openneuro/received/ds003645')

def load_events(sub, run):
    p = DATA_ROOT / f'sub-{sub}' / f'sub-{sub}_task-FacePerception_run-{run}_events.tsv'
    if not p.exists(): return pd.DataFrame()
    ev = pd.read_csv(p, sep='\t')
    face_rows = ev[ev['event_type'] == 'show_face'][['onset','face_type']].copy()
    face_rows['is_face']      = face_rows['face_type'].isin(['famous_face','unfamiliar_face'])
    face_rows['is_scrambled'] = face_rows['face_type'] == 'scrambled_face'
    return face_rows

def label_epochs(ws_arr, we_arr, ev_df, min_stim=1):
    labels, scr_frac = [], []
    for ws, we in zip(ws_arr, we_arr):
        in_w = ev_df[(ev_df['onset'] >= ws) & (ev_df['onset'] < we)]
        nf, ns = int(in_w['is_face'].sum()), int(in_w['is_scrambled'].sum())
        nt = nf + ns
        if nt < min_stim:
            labels.append('no_stim'); scr_frac.append(np.nan)
        else:
            frac = ns / nt; scr_frac.append(frac)
            labels.append('scrambled' if frac >= 0.6 else ('face' if frac <= 0.4 else 'mixed'))
    return np.array(labels), np.array(scr_frac, dtype=float)

label_rows = []
for key, d in records.items():
    sub_id, run_id = key.replace('sub-','').split('_run-')
    ev_df = load_events(sub_id, run_id)
    if ev_df.empty: continue
    n_fif = d['mnps_3d'].shape[0] // 2
    ws = d['window_start'][:n_fif]; we = d['window_end'][:n_fif]
    labels, scr_frac = label_epochs(ws, we, ev_df)
    for i,(lab,sf) in enumerate(zip(labels,scr_frac)):
        label_rows.append({'sub':sub_id,'run':run_id,'epoch_fif_idx':i,
                           'window_start':float(ws[i]),'window_end':float(we[i]),
                           'condition':lab,'scrambled_fraction':float(sf) if np.isfinite(sf) else None})

epoch_labels = pd.DataFrame(label_rows)
print(epoch_labels['condition'].value_counts().to_string())
epoch_labels.to_csv(SAVE_DIR / 'epoch_condition_labels.csv', index=False)
print(f'Saved  shape={epoch_labels.shape}')

condition
face         2131
scrambled     788
mixed         590
no_stim       174
Saved  shape=(3683, 7)


In [19]:
# 7b: Build labeled manifold DataFrame
mnps_rows = []
for key, d in records.items():
    sub_id, run_id = key.replace('sub-','').split('_run-')
    n_fif = d['mnps_3d'].shape[0] // 2
    sub_labels = epoch_labels[(epoch_labels['sub']==sub_id)&(epoch_labels['run']==run_id)].reset_index(drop=True)
    if len(sub_labels)==0: continue
    mnps=d['mnps_3d'][:n_fif]; c9d=d['coords_9d'][:n_fif]
    feat=d['feat_raw'][:n_fif]; fnames=d['feat_names']; qc=d['qc_ok'][:n_fif]
    for i in range(min(n_fif,len(sub_labels))):
        row={'sub':sub_id,'run':run_id,'epoch_fif_idx':i,'qc_ok':bool(qc[i]),
             'condition':sub_labels.at[i,'condition'],
             'scrambled_fraction':sub_labels.at[i,'scrambled_fraction']}
        for j,c in enumerate(['mnps_0','mnps_1','mnps_2']): row[c]=float(mnps[i,j])
        for j,name in enumerate(d['coord_names']): row[f'c9_{name}']=float(c9d[i,j])
        for j,name in enumerate(fnames): row[name]=float(feat[i,j])
        mnps_rows.append(row)
mnps_df = pd.DataFrame(mnps_rows)
mnps_df.to_csv(SAVE_DIR/'labeled_manifold_epochs.csv',index=False)
print(f'shape={mnps_df.shape}')
print(mnps_df.groupby('condition').size().to_string())

shape=(3683, 67)
condition
face         2131
mixed         590
no_stim       174
scrambled     788


In [20]:
# 7c: Centroid distance + permutation test
MNPS_COLS=['mnps_0','mnps_1','mnps_2']
C9D_COLS=[f'c9_{n}' for n in records[next(iter(records))]['coord_names']]
sub_qc=mnps_df[mnps_df['qc_ok']]
face_ep=sub_qc[sub_qc['condition']=='face'][MNPS_COLS].to_numpy(dtype=float)
scr_ep =sub_qc[sub_qc['condition']=='scrambled'][MNPS_COLS].to_numpy(dtype=float)
print(f'Face: {len(face_ep)}, Scrambled: {len(scr_ep)}')
face_centroid=np.nanmean(face_ep,0); scr_centroid=np.nanmean(scr_ep,0)
obs_dist=np.linalg.norm(face_centroid-scr_centroid)
print(f'Face centroid:      {face_centroid.round(4)}')
print(f'Scrambled centroid: {scr_centroid.round(4)}')
print(f'Centroid distance:  {obs_dist:.4f}')
face_9d=sub_qc[sub_qc['condition']=='face'][C9D_COLS].mean()
scr_9d =sub_qc[sub_qc['condition']=='scrambled'][C9D_COLS].mean()
delta_9d=(face_9d-scr_9d).rename(lambda c:c.replace('c9_',''))
print('Face-Scrambled per 9D subcoord:')
print(delta_9d.round(4).to_string())

rng2=np.random.default_rng(0); perm_dists=[]
for _ in range(2000):
    shuf=mnps_df.copy()
    for (_s,_r),grp in shuf.groupby(['sub','run']):
        shuf.loc[grp.index,'condition']=grp['condition'].to_numpy()[rng2.permutation(len(grp))]
    qp=shuf[shuf['qc_ok']]
    fe=qp[qp['condition']=='face'][MNPS_COLS].to_numpy(dtype=float)
    se=qp[qp['condition']=='scrambled'][MNPS_COLS].to_numpy(dtype=float)
    if len(fe)>=5 and len(se)>=5: perm_dists.append(np.linalg.norm(np.nanmean(fe,0)-np.nanmean(se,0)))
perm_arr=np.array(perm_dists); p_perm=np.mean(perm_arr>=obs_dist)
print(f'\nPermutation: obs={obs_dist:.4f} null={perm_arr.mean():.4f}+/-{perm_arr.std():.4f} p={p_perm:.4f}')
contrast_results={'n_face':int(len(face_ep)),'n_scr':int(len(scr_ep)),
    'face_centroid':face_centroid.tolist(),'scr_centroid':scr_centroid.tolist(),
    'centroid_dist':round(float(obs_dist),5),
    'null_mean':round(float(perm_arr.mean()),5),'null_std':round(float(perm_arr.std()),5),
    'p_perm':round(float(p_perm),4),'n_perm':len(perm_arr),
    'delta_9d':delta_9d.round(5).to_dict()}
with open(SAVE_DIR/'face_scrambled_contrast.json','w') as fh: json.dump(contrast_results,fh,indent=2)
print('Saved face_scrambled_contrast.json')

Face: 2131, Scrambled: 788
Face centroid:      [0.     0.     0.0436]
Scrambled centroid: [0.    0.    0.042]
Centroid distance:  0.0016
Face-Scrambled per 9D subcoord:
m_a    0.0000
m_e    0.0000
m_o    0.0000
d_n    0.0000
d_l    0.0000
d_s    0.0000
e_e    0.0000
e_s    0.0000
e_m    0.0787



Permutation: obs=0.0016 null=0.0007+/-0.0005 p=0.0760
Saved face_scrambled_contrast.json


In [21]:
# 7d: Per-band contrast + plots
from scipy.stats import ttest_ind
brows=[]
for mod in ['meg','eeg']:
    for band in ['delta','theta','alpha','beta','gamma']:
        col=f'{mod}_{band}'
        if col not in mnps_df.columns: continue
        fv=sub_qc[sub_qc['condition']=='face'][col].dropna().to_numpy(dtype=float)
        sv=sub_qc[sub_qc['condition']=='scrambled'][col].dropna().to_numpy(dtype=float)
        if len(fv)<5 or len(sv)<5: continue
        lf=np.log10(fv+1e-40); ls=np.log10(sv+1e-40)
        t,p=ttest_ind(lf,ls,equal_var=False)
        d=(lf.mean()-ls.mean())/np.sqrt((lf.std()**2+ls.std()**2)/2)
        brows.append({'modality':mod,'band':band,'diff_log10':round(lf.mean()-ls.mean(),4),
                      'cohens_d':round(d,4),'t_stat':round(t,3),'p_ttest':round(p,5),
                      'n_face':len(lf),'n_scr':len(ls)})
band_df=pd.DataFrame(brows)
band_df.to_csv(SAVE_DIR/'face_scrambled_band_contrast.csv',index=False)
print(band_df.to_string(index=False))

bands_o=['delta','theta','alpha','beta','gamma']
fig,axes=plt.subplots(1,3,figsize=(15,5))
axes[0].hist(perm_arr,bins=50,color='steelblue',alpha=0.7,label='null')
axes[0].axvline(obs_dist,color='crimson',lw=2.5,label=f'obs ({obs_dist:.3f})')
axes[0].set_title(f'Permutation test  p={p_perm:.3f}'); axes[0].legend(fontsize=8)
x=np.arange(5); w=0.35
for im,(mod,col2) in enumerate([('meg','steelblue'),('eeg','darkorange')]):
    sb=band_df[band_df['modality']==mod].set_index('band')
    ds=[float(sb.at[b,'cohens_d']) if b in sb.index else np.nan for b in bands_o]
    axes[1].bar(x+im*w-w/2,ds,w,label=mod.upper(),color=col2,alpha=0.8)
axes[1].axhline(0,color='k',lw=0.7); axes[1].set_xticks(x); axes[1].set_xticklabels(bands_o)
axes[1].set_ylabel("Cohen's d"); axes[1].set_title('Band contrast face-scrambled'); axes[1].legend()
ff=sub_qc[sub_qc['condition']=='face'][['mnps_0','mnps_1']].dropna().to_numpy()
fs=sub_qc[sub_qc['condition']=='scrambled'][['mnps_0','mnps_1']].dropna().to_numpy()
if len(ff): axes[2].scatter(ff[:,0],ff[:,1],s=5,color='steelblue',alpha=0.4,label='face')
if len(fs): axes[2].scatter(fs[:,0],fs[:,1],s=5,color='crimson',alpha=0.4,label='scrambled')
axes[2].scatter(*face_centroid[:2],s=120,marker='*',color='navy',label='face ctr')
axes[2].scatter(*scr_centroid[:2],s=120,marker='*',color='darkred',label='scr ctr')
axes[2].set_title('MNPS-3D face vs scrambled'); axes[2].legend(markerscale=2,fontsize=8)
fig.suptitle('Face vs Scrambled --- ds003645 pilot'); fig.tight_layout()
fig.savefig(FIG_DIR/'face_scrambled_contrast.png',dpi=150); plt.close(fig)
print('Saved face_scrambled_contrast.png')

modality  band  diff_log10  cohens_d  t_stat  p_ttest  n_face  n_scr
     eeg delta      0.0036    0.0052   0.126  0.89983    2131    788
     eeg theta     -0.0202   -0.0489  -1.177  0.23921    2131    788
     eeg alpha      0.0050    0.0133   0.320  0.74884    2131    788
     eeg  beta      0.0115    0.0482   1.153  0.24909    2131    788
     eeg gamma      0.0114    0.0492   1.173  0.24118    2131    788


Saved face_scrambled_contrast.png


## 8 - Per-run Procrustes (corrected)

Global PCA mixes between-subject variance. Per-run analysis is fairer.

In [22]:
from scipy.spatial import procrustes as proc_scipy
perrun_rows=[]
for (sub,run),grp in all_df.groupby(['sub','run']):
    fg=grp[(grp['source']=='fif')&(grp['qc_ok']==1)]
    if len(fg)<10: continue
    ez=[c+'_z' for c in EEG_FEAT_COLS if c+'_z' in fg.columns]
    mz=[c+'_z' for c in MEG_FEAT_COLS  if c+'_z' in fg.columns]
    em=fg[ez].to_numpy(dtype=float); mm=fg[mz].to_numpy(dtype=float)
    v=np.isfinite(em).all(1)&np.isfinite(mm).all(1)
    if v.sum()<6: continue
    ec=em[v]; mc=mm[v]; nc=min(3,ec.shape[1],mc.shape[1],ec.shape[0]-1)
    pe=PCA(nc); pm=PCA(nc)
    e3=pe.fit_transform(ec); m3=pm.fit_transform(mc)
    if e3.shape[1]<3 or m3.shape[1]<3: continue
    def nc2(X): X=X-X.mean(0); n=np.sqrt((X**2).sum()); return X/n if n>0 else X
    try: _,_,d=proc_scipy(nc2(m3),nc2(e3)); sim=round(1.0-d,4)
    except: sim=np.nan
    perrun_rows.append({'sub':sub,'run':run,'n':int(v.sum()),'procrustes_sim':sim,
                        'eeg_var3':round(float(pe.explained_variance_ratio_.sum()),3),
                        'meg_var3':round(float(pm.explained_variance_ratio_.sum()),3)})
perrun_df=pd.DataFrame(perrun_rows)
perrun_df.to_csv(SAVE_DIR/'perrun_procrustes.csv',index=False)
print(perrun_df[['sub','run','n','procrustes_sim']].to_string(index=False))
m=perrun_df['procrustes_sim'].mean(); s=perrun_df['procrustes_sim'].std()
print(f'Mean+/-SD: {m:.3f}+/-{s:.3f}')

fig,ax=plt.subplots(figsize=(8,4))
for sid in sorted(perrun_df['sub'].unique()):
    r=perrun_df[perrun_df['sub']==sid]
    ax.plot(r['run'].astype(int),r['procrustes_sim'],marker='o',label=f'sub-{sid}')
ax.axhline(perrun_df['procrustes_sim'].mean(),color='k',ls='--',lw=1,label='mean')
ax.set_xlabel('Run'); ax.set_ylabel('Procrustes similarity'); ax.set_ylim(0,1)
ax.set_title('Per-run MEG vs EEG Procrustes --- ds003645'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR/'perrun_procrustes.png',dpi=150); plt.close(fig)
print('Saved perrun_procrustes.png')

sub run   n  procrustes_sim
002   1 121          0.0149
002   2 123          0.0521
002   3 125          0.0510
002   4 122          0.0790
002   5 125          0.0632
002   6 124          0.1267
003   1 122          0.0662
003   2 122          0.0445
003   3 122          0.0251
003   4 124          0.0174
003   5 122          0.0355
003   6 121          0.0420
004   1 125          0.0923
004   2 122          0.1436
004   3 125          0.0483
004   4 123          0.0190
004   5 122          0.0133
004   6 124          0.0124
005   1 123          0.0347
005   2 121          0.0769
005   3 124          0.0604
005   4 122          0.0324
005   5 122          0.0073
005   6 123          0.0230
006   1 121          0.0187
006   2 125          0.0805
006   3 123          0.0584
006   4 120          0.0151
006   5 124          0.0296
006   6 121          0.0180
Mean+/-SD: 0.047+/-0.034


Saved perrun_procrustes.png


## 9 - EEG-from-FIF vs EEG-from-.set consistency

In [23]:
consist_rows=[]
for key,d in records.items():
    sub_id,run_id=key.replace('sub-','').split('_run-')
    n_fif=d['feat_raw'].shape[0]//2
    ws_fif=d['window_start'][:n_fif]; ws_set=d['window_start'][n_fif:]
    feat=d['feat_raw']; fnames=d['feat_names']
    for band in ['delta','theta','alpha','beta','gamma']:
        col=f'eeg_{band}'
        if col not in fnames: continue
        ci=fnames.index(col); fv=feat[:n_fif,ci]; sv=feat[n_fif:,ci]
        mf,ms=[],[]
        for i,wsf in enumerate(ws_fif):
            diffs=np.abs(ws_set-wsf); j=int(np.argmin(diffs))
            if diffs[j]<0.5: mf.append(fv[i]); ms.append(sv[j])
        if len(mf)<10: continue
        lf2=np.log10(np.array(mf,dtype=float)+1e-30)
        ls2=np.log10(np.array(ms,dtype=float)+1e-30)
        v=np.isfinite(lf2)&np.isfinite(ls2)
        if v.sum()<10: continue
        r,p=pearsonr(lf2[v],ls2[v])
        consist_rows.append({'sub':sub_id,'run':run_id,'band':band,
                              'n':int(v.sum()),'r':round(r,4),'p':round(p,6)})
consist_df=pd.DataFrame(consist_rows)
consist_df.to_csv(SAVE_DIR/'eeg_fif_vs_set_consistency.csv',index=False)
sc=consist_df.groupby('band')['r'].agg(['mean','std','min','max']).round(3)
print('EEG FIF vs .set consistency (Pearson r):'); print(sc.to_string())

fig,ax=plt.subplots(figsize=(8,4))
ax.bar(sc.index,sc['mean'],yerr=sc['std'],capsize=4,color='teal',alpha=0.8)
ax.axhline(1,color='k',ls='--',lw=0.8)
ax.set_ylabel('Pearson r  (FIF vs .set EEG)'); ax.set_ylim(0,1.1)
ax.set_title('EEG consistency: FIF-embedded vs standalone .set')
fig.tight_layout(); fig.savefig(FIG_DIR/'eeg_fif_vs_set_consistency.png',dpi=150); plt.close(fig)
print('Saved eeg_fif_vs_set_consistency.png')

EEG FIF vs .set consistency (Pearson r):
        mean    std    min  max
band                           
alpha  0.881  0.313 -0.072  1.0
beta   0.878  0.334 -0.387  1.0
delta  0.875  0.330 -0.142  1.0
gamma  0.893  0.291 -0.136  1.0
theta  0.878  0.318 -0.048  1.0
Saved eeg_fif_vs_set_consistency.png


## 10 - Final summary

In [24]:
full_summary={'dataset':'ds003645','task':TASK,'subjects':SUBS,'n_files':len(records),
    'meg_eeg_corr':{f:round(corr_df[corr_df['feature']==f]['pearson_r'].mean(),3) for f in BANDS if f in corr_df['feature'].values},
    'temporal_null':{r['feature']:{'r_obs':r['r_observed'],'null':r['null_mean'],'p':r['p_null']} for _,r in null_df.iterrows()},
    'per_run_procrustes':{'mean':round(float(perrun_df['procrustes_sim'].mean()),4),'std':round(float(perrun_df['procrustes_sim'].std()),4)},
    'face_scrambled':contrast_results,
    'eeg_consistency':{b:round(float(consist_df[consist_df['band']==b]['r'].mean()),3) for b in consist_df['band'].unique()},
}
with open(SAVE_DIR/'full_analysis_summary.json','w') as fh: json.dump(full_summary,fh,indent=2)
print(json.dumps(full_summary,indent=2))

{
  "dataset": "ds003645",
  "task": "FacePerception",
  "subjects": [
    "002",
    "003",
    "004",
    "005",
    "006"
  ],
  "n_files": 30,
  "meg_eeg_corr": {
    "delta": 0.099,
    "theta": 0.031,
    "alpha": 0.231,
    "beta": 0.147,
    "gamma": 0.214
  },
  "temporal_null": {
    "delta": {
      "r_obs": 0.0053,
      "null": 0.0051,
      "p": 0.345
    },
    "theta": {
      "r_obs": 0.0088,
      "null": 0.0031,
      "p": 0.235
    },
    "alpha": {
      "r_obs": 0.0532,
      "null": 0.0025,
      "p": 0.015
    },
    "beta": {
      "r_obs": 0.0671,
      "null": 0.0033,
      "p": 0.015
    },
    "gamma": {
      "r_obs": 0.1548,
      "null": 0.0023,
      "p": 0.0
    }
  },
  "per_run_procrustes": {
    "mean": 0.0467,
    "std": 0.0337
  },
  "face_scrambled": {
    "n_face": 2131,
    "n_scr": 788,
    "face_centroid": [
      0.0,
      0.0,
      0.04361164134785932
    ],
    "scr_centroid": [
      0.0,
      0.0,
      0.04203776642798356
    ],
    